<a href="https://colab.research.google.com/github/khristianenggar367-tech/Precipitation-Adjusted-Trends-in-Tropical-Drylands/blob/main/Kappa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd

# 1. Konfigurasi File Input & Output
CSV_FILE = 'Data_Iklim_METO_2020_2024.csv'
OUTPUT_DIR = 'SWAT_Climate_METO'

# Metadata Lokasi Stasiun
LAT = -10.000
LON = 123.000
ELEV = 100.0


def convert_csv_to_swat():
  if not os.path.exists(CSV_FILE):
    print(f'Error: File {CSV_FILE} tidak ditemukan!')
    return

  os.makedirs(OUTPUT_DIR, exist_ok=True)

  # Membaca data CSV
  df = pd.read_csv(CSV_FILE)
  df['date'] = pd.to_datetime(df['date'])

  # Memastikan rentang tanggal lengkap (1 Jan 2020 s.d. 31 Des 2024 -> 1.827 hari)
  full_dates = pd.date_range(start='2020-01-01', end='2024-12-31', freq='D')
  df = df.set_index('date').reindex(full_dates)

  # Mengisi tanggal 31-12-2024 yang missing menggunakan ffill
  df = df.ffill().reset_index().rename(columns={'index': 'date'})
  df['YYYYMMDD'] = df['date'].dt.strftime('%Y%m%d')

  total_days = len(df)
  nbyr = round(total_days / 365.25)

  # Header Standar SWAT+
  header_line = (
      f'nbyr={nbyr} tstep=daily lat={LAT:.3f} lon={LON:.3f} elev={ELEV:.1f}'
  )

  print(f'Memproses {total_days} hari data ({nbyr} tahun)...')

  # Mapping variabel iklim ke file output
  weather_vars = {
      'meto_pcp.txt': ['pcp_mm'],
      'meto_tmp.txt': ['tmax_C', 'tmin_C'],
      'meto_slr.txt': ['solar_mj'],
      'meto_rh.txt': ['rh_fraction'],
      'meto_wnd.txt': ['wind_ms'],
      'meto_pet.txt': ['pet_mm'],
  }

  # Generate File Data Iklim Harian (.txt)
  for fname, cols in weather_vars.items():
    filepath = os.path.join(OUTPUT_DIR, fname)
    with open(filepath, 'w', encoding='utf-8') as f:
      f.write(header_line + '\n')
      for _, row in df.iterrows():
        val_str = '  '.join([f'{row[c]:.3f}' for c in cols])
        f.write(f"{row['YYYYMMDD']}  {val_str}\n")
    print(f' -> Berhasil membuat {fname}')

  # Generate File Indeks Stasiun untuk SWAT+ Editor (CSV/TXT Indeks)
  index_files = {
      'pcp.txt': 'meto_pcp.txt',
      'tmp.txt': 'meto_tmp.txt',
      'slr.txt': 'meto_slr.txt',
      'rh.txt': 'meto_rh.txt',
      'wnd.txt': 'meto_wnd.txt',
      'pet.txt': 'meto_pet.txt',
  }

  for idx_fname, data_fname in index_files.items():
    idx_path = os.path.join(OUTPUT_DIR, idx_fname)
    with open(idx_path, 'w', encoding='utf-8') as f:
      f.write('filename,lat,lon,elev\n')
      f.write(f'{data_fname},{LAT:.3f},{LON:.3f},{ELEV:.1f}\n')

  print(
      f"\n[SUKSES] Seluruh file SWAT siap dieksekusi di folder: '{OUTPUT_DIR}/'"
  )


if __name__ == '__main__':
  convert_csv_to_swat()

Memproses 1827 hari data (5 tahun)...
 -> Berhasil membuat meto_pcp.txt
 -> Berhasil membuat meto_tmp.txt
 -> Berhasil membuat meto_slr.txt
 -> Berhasil membuat meto_rh.txt
 -> Berhasil membuat meto_wnd.txt
 -> Berhasil membuat meto_pet.txt

[SUKSES] Seluruh file SWAT siap dieksekusi di folder: 'SWAT_Climate_METO/'


In [ ]:
import pandas as pd

# Baca file usersoil asli Anda
df = pd.read_csv('usersoil.csv')

# Hitung fraksi Pasir (SAND) & Batuan (ROCK)
df['SAND1'] = 100.0 - (df['CLAY1'] + df['SILT1'])
df['ROCK1'] = 0.0

# Perbaiki parameter retensi air & kepadatan agar aman dari overflow
df['SOL_AWC1'] = df['SOL_AWC1'].apply(lambda x: max(x * 2.5, 0.14) if x < 0.10 else x)
df['SOL_BD1'] = df['SOL_BD1'].apply(lambda x: max(x, 1.20))

# Tambahkan parameter fisik tanah standar SWAT+ yang hilang
df['SOL_ALB1'] = 0.13      # Soil albedo
df['SOL_USLE_K1'] = 0.28   # USLE K factor
df['SOL_EC1'] = 0.0        # Electrical conductivity

# Reorder kolom sesuai skema standar SWAT+
cols = [
    'SOIL_ID', 'SNAM', 'HYDGRP', 'NLAYERS', 'SOL_Z1', 'SOL_BD1', 'SOL_AWC1',
    'SOL_K1', 'SOL_CBN1', 'CLAY1', 'SILT1', 'SAND1', 'ROCK1', 'SOL_ALB1',
    'SOL_USLE_K1', 'SOL_EC1'
]

df_complete = df[cols]
df_complete.to_csv('usersoil_complete.csv', index=False)
print("File usersoil_complete.csv dengan 18 kolom standar SWAT+ berhasil dibuat!")

File usersoil_complete.csv dengan 18 kolom standar SWAT+ berhasil dibuat!


In [1]:
import os
import numpy as np
import pandas as pd
from osgeo import gdal
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# ==============================================================================
# PART 1: DATA ACCOUNTING & GAP-FILLING VERIFICATION
# ==============================================================================
def verify_table4_accounting(raw_raster_folder, filled_raster_folder, start_year=1990, end_year=2024, nodata_val=-9999.0):
    """
    Menghitung jumlah piksel valid mentah, piksel terinterpolasi, dan sisa NoData
    per tahun dari 1990 hingga 2024 untuk memverifikasi cakupan 100.00%.
    """
    print("=" * 75)
    print("1. MEMULAI DATA ACCOUNTING SPASIAL PRE- DAN POST-GAP FILLING")
    print("=" * 75)

    total_domain_pixels = 2189883  # Total domain grid DAS Noelmina
    accounting_data = []

    for year in range(start_year, end_year + 1):
        # Sesuaikan format nama file raster
        raw_path = os.path.join(raw_raster_folder, f"NDVI_{year}_Raw.tif")
        filled_path = os.path.join(filled_raster_folder, f"NDVI_{year}_Filled.tif")

        # Buka raster jika file tersedia
        if os.path.exists(raw_path) and os.path.exists(filled_path):
            ds_raw = gdal.Open(raw_path)
            ds_filled = gdal.Open(filled_path)

            arr_raw = ds_raw.GetRasterBand(1).ReadAsArray()
            arr_filled = ds_filled.GetRasterBand(1).ReadAsArray()

            # Hitung piksel valid
            raw_valid = np.sum((arr_raw != nodata_val) & (~np.isnan(arr_raw)))
            interpolated = total_domain_pixels - raw_valid
            remaining_nodata = np.sum((arr_filled == nodata_val) | (np.isnan(arr_filled)))
            coverage = ((total_domain_pixels - remaining_nodata) / total_domain_pixels) * 100.0

            accounting_data.append({
                'Year': year,
                'Baseline Raw Valid Pixels': raw_valid,
                'Successfully Interpolated Pixels': interpolated,
                'Remaining NoData Pixels': remaining_nodata,
                'Final Valid Coverage (%)': f"{coverage:.2f}%"
            })
        else:
            # Simulasi fallback untuk pengujian dummy jika file fisik belum terhubung
            pass

    df_accounting = pd.DataFrame(accounting_data)
    print(df_accounting.to_string(index=False))
    return df_accounting

# ==============================================================================
# PART 2: ANALYSIS SENSITIVITAS COHEN'S KAPPA
# ==============================================================================
def calculate_sensitivity_kappa(raw_trend_raster, filled_trend_raster, nodata_val=-9999.0):
    """
    Membandingkan peta tren klasifikasi Mann-Kendall + Sen's Slope (Raw vs Gap-Filled)
    Kategori tren:
      1 = Significant Greening (Z > +1.96, p < 0.05)
      2 = Significant Browning (Z < -1.96, p < 0.05)
      0 = Baseline Stability / No Trend (|Z| <= 1.96)
    """
    print("\n" + "=" * 75)
    print("2. MEMULAI UJI SENSITIVITAS & KESEPAKATAN SPASIAL COHEN'S KAPPA (κ)")
    print("=" * 75)

    ds_raw = gdal.Open(raw_trend_raster)
    ds_filled = gdal.Open(filled_trend_raster)

    if ds_raw is None or ds_filled is None:
        raise FileNotFoundError("File GeoTIFF tren tidak ditemukan. Periksa path input Anda.")

    arr_raw = ds_raw.GetRasterBand(1).ReadAsArray()
    arr_filled = ds_filled.GetRasterBand(1).ReadAsArray()

    # Filter piksel valid (domain vegetasi teruji N = 2.187.693 piksel)
    valid_mask = (
        (arr_raw != nodata_val) & (~np.isnan(arr_raw)) &
        (arr_filled != nodata_val) & (~np.isnan(arr_filled))
    )

    vec_raw = arr_raw[valid_mask].astype(int)
    vec_filled = arr_filled[valid_mask].astype(int)

    total_n = len(vec_raw)

    # Hitung Matriks Konfusi
    labels = [0, 1, 2]  # 0: No Trend, 1: Greening, 2: Browning
    cm = confusion_matrix(vec_raw, vec_filled, labels=labels)

    # Hitung Persentase Akurasi & Cohen's Kappa
    overall_agreement = (np.trace(cm) / total_n) * 100.0
    kappa_val = cohen_kappa_score(vec_raw, vec_filled)

    # Tampilkan Hasil Statistik
    print(f"Total Valid Vegetation Pixels Tested (N) : {total_n:,} piksel")
    print(f"Overall Trend Classification Agreement   : {overall_agreement:.1f}%")
    print(f"Cohen's Kappa Coefficient (κ)            : {kappa_val:.3f}")
    print("-" * 75)
    print("Matriks Konfusi Spasial (Baris: Raw Uninterpolated, Kolom: Gap-Filled):")
    df_cm = pd.DataFrame(cm, index=['No Trend (0)', 'Greening (1)', 'Browning (2)'],
                             columns=['No Trend (0)', 'Greening (1)', 'Browning (2)'])
    print(df_cm)

    # Verifikasi kesesuaian
    print("\n--- STATUS VERIFIKASI ---")
    if abs(overall_agreement - 97.8) < 0.5 and abs(kappa_val - 0.962) < 0.01:
        print("[SUCCESS] Hasil perhitungan PERSATU secara presisi")
    else:
        print(f"[INFO] Nilai terhitung: Agreement = {overall_agreement:.1f}%, Kappa = {kappa_val:.3f}")

# ==============================================================================
# MAIN EXECUTION BLOCK
# ==============================================================================
if __name__ == "__main__":
    # Ganti path folder dan file sesuai dengan lokasi GeoTIFF di komputer
    RAW_RASTER_DIR = r"H:\MAST_Residu\NDVI_Raw_Annual"
    FILLED_RASTER_DIR = r"H:\MAST_Residu\NDVI_Filled_Annual"

    RAW_TREND_GEOTIFF = r"H:\MAST_Residu\NDVI_Raw_MannKendall_Trend.tif"
    FILLED_TREND_GEOTIFF = r"H:\MAST_Residu\NDVI_Filled_MannKendall_Trend.tif"

    # 1. Jalankan accounting
    # verify_table4_accounting(RAW_RASTER_DIR, FILLED_RASTER_DIR)

    # 2. Jalankan perhitungan Kappa
    # calculate_sensitivity_kappa(RAW_TREND_GEOTIFF, FILLED_TREND_GEOTIFF)



# Bagian Baru